<a href="https://colab.research.google.com/github/turxannbiyev13/Polars-movie-analysis/blob/main/polars_movie_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring and Analyzing Movie Ratings with Polars: Project



| File        | Column     | Description                                                                 |
|------------|-----------|------------------------------------------------------------------------------|
| movies.csv | movieId   | ID number that uniquely identifies each movie                                |
| movies.csv | title     | Name of the movie (may include year in parentheses)                          |
| movies.csv | genres    | Pipe-separated list of genres (e.g., Comedy|Drama)                           |
| ratings.csv| userId    | ID number that uniquely identifies each user                                 |
| ratings.csv| movieId   | ID number that uniquely identifies each movie                                |
| ratings.csv| rating    | Rating score from 0.5 to 5.0 (half-star increments allowed)                  |
| ratings.csv| timestamp | Unix timestamp when the rating was given                                     |
| tags.csv   | userId    | ID number that uniquely identifies each user                                 |
| tags.csv   | movieId   | ID number that uniquely identifies each movie                                |
| tags.csv   | tag       | User-generated tag (string, e.g., 'funny')                                    |
| tags.csv   | timestamp | Unix timestamp when the tag was added                                        |


## Step 1: Load and inspect data

In [62]:
import polars as pl

In [63]:
movies = pl.read_csv('movies.csv')
ratings = pl.read_csv('ratings.csv')
tags = pl.read_csv('tags.csv')

## Step 2: Preprocess data

In [64]:
print(movies.head())
print(ratings.head())
print(tags.head())

shape: (5, 3)
┌─────────┬─────────────────────────────────┬─────────────────────────────────┐
│ movieId ┆ title                           ┆ genres                          │
│ ---     ┆ ---                             ┆ ---                             │
│ i64     ┆ str                             ┆ str                             │
╞═════════╪═════════════════════════════════╪═════════════════════════════════╡
│ 1       ┆ Toy Story (1995)                ┆ Adventure|Animation|Children|C… │
│ 2       ┆ Jumanji (1995)                  ┆ Adventure|Children|Fantasy      │
│ 3       ┆ Grumpier Old Men (1995)         ┆ Comedy|Romance                  │
│ 4       ┆ Waiting to Exhale (1995)        ┆ Comedy|Drama|Romance            │
│ 5       ┆ Father of the Bride Part II (1… ┆ Comedy                          │
└─────────┴─────────────────────────────────┴─────────────────────────────────┘
shape: (5, 4)
┌────────┬─────────┬────────┬───────────┐
│ userId ┆ movieId ┆ rating ┆ timestamp │
│ ---   

In [65]:
ratings = ratings.with_columns([
    pl.col("userId").cast(pl.Int64),
    pl.col("movieId").cast(pl.Int64),
    pl.col("timestamp").cast(pl.Int64) ])

In [66]:
movies = movies.unique()
ratings = ratings.unique()
tags = tags.unique()

In [67]:
movies = movies.fill_null("Məlumat yoxdur")
tags = tags.fill_null("Teq yoxdur")

## [A] Easy

### A1: Verify that values in the 'rating' column in the ratings table are sensible (i.e., ranges from 0.5 to 5.0). Print the min and max values.

In [68]:
min_xal = ratings["rating"].min()
max_xal = ratings["rating"].max()
print("Minimum xal:", min_xal, "| Maksimum xal:", max_xal)

Minimum xal: 0.5 | Maksimum xal: 5.0


### A2: Compute and print the count of ratings for each score (0.5 through 5.0) in a table format.

In [69]:
xal_saylari = ratings.group_by("rating").count()
xal_saylari = xal_saylari.sort("rating")
print(xal_saylari)

shape: (10, 2)
┌────────┬───────┐
│ rating ┆ count │
│ ---    ┆ ---   │
│ f64    ┆ u32   │
╞════════╪═══════╡
│ 0.5    ┆ 1370  │
│ 1.0    ┆ 2811  │
│ 1.5    ┆ 1791  │
│ 2.0    ┆ 7551  │
│ 2.5    ┆ 5550  │
│ 3.0    ┆ 20047 │
│ 3.5    ┆ 13136 │
│ 4.0    ┆ 26818 │
│ 4.5    ┆ 8551  │
│ 5.0    ┆ 13211 │
└────────┴───────┘


/tmp/ipykernel_6261/718286184.py:1: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  xal_saylari = ratings.group_by("rating").count()


### A3: Compute and print the average rating per genre (explode the genres_list column and group by genre).

In [70]:
movies_parcalanmis = movies.with_columns(pl.col("genres").str.split("|"))
movies_janrlar = movies_parcalanmis.explode("genres")

In [71]:
janr_ve_reytinq = movies_janrlar.join(ratings, on="movieId")

In [72]:
janr_ortalamasi = janr_ve_reytinq.group_by("genres").agg(pl.col("rating").mean())
janr_ortalamasi = janr_ortalamasi.sort("rating", descending=True)
print(janr_ortalamasi)

shape: (20, 2)
┌─────────────┬──────────┐
│ genres      ┆ rating   │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ Film-Noir   ┆ 3.920115 │
│ War         ┆ 3.808294 │
│ Documentary ┆ 3.797785 │
│ Crime       ┆ 3.658294 │
│ Drama       ┆ 3.656184 │
│ …           ┆ …        │
│ Sci-Fi      ┆ 3.455721 │
│ Action      ┆ 3.447984 │
│ Children    ┆ 3.412956 │
│ Comedy      ┆ 3.384721 │
│ Horror      ┆ 3.258195 │
└─────────────┴──────────┘


### A4: Compute and print the min, max, and mean number of ratings per user.

In [73]:
user_rating_counts = ratings.group_by("userId").agg(
    pl.len().alias("rating_count")
)

In [74]:
stats = user_rating_counts.select(
    pl.col("rating_count").min().alias("min_ratings"),
    pl.col("rating_count").max().alias("max_ratings"),
    pl.col("rating_count").mean().alias("mean_ratings")
)

print(stats)

shape: (1, 3)
┌─────────────┬─────────────┬──────────────┐
│ min_ratings ┆ max_ratings ┆ mean_ratings │
│ ---         ┆ ---         ┆ ---          │
│ u32         ┆ u32         ┆ f64          │
╞═════════════╪═════════════╪══════════════╡
│ 20          ┆ 2698        ┆ 165.304918   │
└─────────────┴─────────────┴──────────────┘


### A5: Find and print the top 5 users with the most ratings.




In [75]:
top_5_users = (
    ratings.group_by("userId")
    .agg(pl.len().alias("rating_count"))
    .sort("rating_count", descending=True)
    .head(5)
)

In [76]:
print(top_5_users)

shape: (5, 2)
┌────────┬──────────────┐
│ userId ┆ rating_count │
│ ---    ┆ ---          │
│ i64    ┆ u32          │
╞════════╪══════════════╡
│ 414    ┆ 2698         │
│ 599    ┆ 2478         │
│ 474    ┆ 2108         │
│ 448    ┆ 1864         │
│ 274    ┆ 1346         │
└────────┴──────────────┘


## [B] Medium

### B1: Compute and print the average rating of the top 10 most frequently rated movies vs. the bottom 10 (movies with the fewest ratings, but at least 5 ratings).

In [77]:
movie_stats = ratings.group_by("movieId").agg(
    pl.len().alias("rating_count"),
    pl.col("rating").mean().alias("avg_rating"))

In [78]:
top_10_movies = movie_stats.sort("rating_count", descending=True).head(10)
top_10_avg = top_10_movies.select(pl.col("avg_rating").mean()).item()

In [79]:
bottom_10_movies = (
    movie_stats.filter(pl.col("rating_count") >= 5)
    .sort("rating_count", descending=False)
    .head(10)
)
bottom_10_avg = bottom_10_movies.select(pl.col("avg_rating").mean()).item()

In [80]:
print(f"Top 10 ən çox reytinq alan filmlərin ümumi ortalaması: {top_10_avg:.2f}")
print(f"Bottom 10 (ən az 5 reytinqi olan) filmlərin ümumi ortalaması: {bottom_10_avg:.2f}")

Top 10 ən çox reytinq alan filmlərin ümumi ortalaması: 4.14
Bottom 10 (ən az 5 reytinqi olan) filmlərin ümumi ortalaması: 3.22


### B2: Compute and print a table showing the distribution of the number of ratings per movie.

In [81]:
movie_rating_counts = ratings.group_by("movieId").agg(
    pl.len().alias("rating_count"))

In [82]:
rating_distribution = (
    movie_rating_counts.group_by("rating_count")
    .agg(pl.len().alias("movie_count"))
    .sort("rating_count")
)

In [83]:
print(rating_distribution)

shape: (177, 2)
┌──────────────┬─────────────┐
│ rating_count ┆ movie_count │
│ ---          ┆ ---         │
│ u32          ┆ u32         │
╞══════════════╪═════════════╡
│ 1            ┆ 3446        │
│ 2            ┆ 1298        │
│ 3            ┆ 800         │
│ 4            ┆ 530         │
│ 5            ┆ 382         │
│ …            ┆ …           │
│ 278          ┆ 1           │
│ 279          ┆ 1           │
│ 307          ┆ 1           │
│ 317          ┆ 1           │
│ 329          ┆ 1           │
└──────────────┴─────────────┘


### B3: Print the top 20 most frequently rated movies (display their id, title, and rating count).

In [84]:
film_statistikasi = ratings.group_by("movieId").agg(
    pl.len().alias("rey_sayi")
)

In [85]:
adli_filmler = film_statistikasi.join(movies, on="movieId")

In [86]:
top_20_filmler = adli_filmler.select(["movieId", "title", "rey_sayi"])
top_20_filmler = top_20_filmler.sort("rey_sayi", descending=True).head(20)
print(top_20_filmler)

shape: (20, 3)
┌─────────┬─────────────────────────────────┬──────────┐
│ movieId ┆ title                           ┆ rey_sayi │
│ ---     ┆ ---                             ┆ ---      │
│ i64     ┆ str                             ┆ u32      │
╞═════════╪═════════════════════════════════╪══════════╡
│ 356     ┆ Forrest Gump (1994)             ┆ 329      │
│ 318     ┆ Shawshank Redemption, The (199… ┆ 317      │
│ 296     ┆ Pulp Fiction (1994)             ┆ 307      │
│ 593     ┆ Silence of the Lambs, The (199… ┆ 279      │
│ 2571    ┆ Matrix, The (1999)              ┆ 278      │
│ …       ┆ …                               ┆ …        │
│ 47      ┆ Seven (a.k.a. Se7en) (1995)     ┆ 203      │
│ 780     ┆ Independence Day (a.k.a. ID4) … ┆ 202      │
│ 150     ┆ Apollo 13 (1995)                ┆ 201      │
│ 1198    ┆ Raiders of the Lost Ark (India… ┆ 200      │
│ 4993    ┆ Lord of the Rings: The Fellows… ┆ 198      │
└─────────┴─────────────────────────────────┴──────────┘


### B4: Compute and print the total number of unique genres and list the top 10 most common genres with their counts.

In [87]:
parcalanmis_janrlar = movies.select(
    pl.col("genres").str.split("|")
).explode("genres")

In [88]:
unikal_janr_sayi = parcalanmis_janrlar.select(pl.col("genres").n_unique()).item()
print(f"Ümumi unikal janr sayı: {unikal_janr_sayi}\n")

Ümumi unikal janr sayı: 20



In [89]:
top_10_janr = (
    parcalanmis_janrlar.group_by("genres")
    .agg(pl.len().alias("film_sayi"))
    .sort("film_sayi", descending=True)
    .head(10)
)

print("Ən çox rast gəlinən top 10 janr:")
print(top_10_janr)

Ən çox rast gəlinən top 10 janr:
shape: (10, 2)
┌───────────┬───────────┐
│ genres    ┆ film_sayi │
│ ---       ┆ ---       │
│ str       ┆ u32       │
╞═══════════╪═══════════╡
│ Drama     ┆ 4361      │
│ Comedy    ┆ 3756      │
│ Thriller  ┆ 1894      │
│ Action    ┆ 1828      │
│ Romance   ┆ 1596      │
│ Adventure ┆ 1263      │
│ Crime     ┆ 1199      │
│ Sci-Fi    ┆ 980       │
│ Horror    ┆ 978       │
│ Fantasy   ┆ 779       │
└───────────┴───────────┘


### B5: For each genre, compute and print the number of movies and average rating in a sorted table.

In [90]:
janrlar_df = movies.select(
    "movieId",
    pl.col("genres").str.split("|")
).explode("genres")

In [91]:
janr_reytinqləri = ratings.join(janrlar_df, on="movieId", how="inner")

In [92]:
janr_analizi = (
    janr_reytinqləri.group_by("genres")
    .agg(
        pl.col("movieId").n_unique().alias("film_sayi"),
        pl.col("rating").mean().alias("orta_reytinq")
    )
    .sort("film_sayi", descending=True) # Filmlərin sayına görə azalan sıra ilə düzürük
)

In [93]:
print(janr_analizi)

shape: (20, 3)
┌────────────────────┬───────────┬──────────────┐
│ genres             ┆ film_sayi ┆ orta_reytinq │
│ ---                ┆ ---       ┆ ---          │
│ str                ┆ u32       ┆ f64          │
╞════════════════════╪═══════════╪══════════════╡
│ Drama              ┆ 4349      ┆ 3.656184     │
│ Comedy             ┆ 3753      ┆ 3.384721     │
│ Thriller           ┆ 1889      ┆ 3.493706     │
│ Action             ┆ 1828      ┆ 3.447984     │
│ Romance            ┆ 1591      ┆ 3.506511     │
│ …                  ┆ …         ┆ …            │
│ Musical            ┆ 333       ┆ 3.563678     │
│ Western            ┆ 167       ┆ 3.583938     │
│ IMAX               ┆ 158       ┆ 3.618335     │
│ Film-Noir          ┆ 85        ┆ 3.920115     │
│ (no genres listed) ┆ 34        ┆ 3.489362     │
└────────────────────┴───────────┴──────────────┘


## [C] Hard

### C1: Compute and print the average number of tags per movie, and show the min and max tags per movie.

In [94]:
film_teq_saylari = tags.group_by("movieId").agg(
    pl.len().alias("teq_sayi")
)

In [95]:
teq_statistikasi = film_teq_saylari.select(
    pl.col("teq_sayi").min().alias("min_teq"),
    pl.col("teq_sayi").max().alias("max_teq"),
    pl.col("teq_sayi").mean().alias("orta_teq")
)

In [96]:
print(teq_statistikasi)

shape: (1, 3)
┌─────────┬─────────┬──────────┐
│ min_teq ┆ max_teq ┆ orta_teq │
│ ---     ┆ ---     ┆ ---      │
│ u32     ┆ u32     ┆ f64      │
╞═════════╪═════════╪══════════╡
│ 1       ┆ 181     ┆ 2.342875 │
└─────────┴─────────┴──────────┘


### C2: Print the top 20 tags that are applied most frequently (display the tag and count).

In [97]:
top_20_teq = (
    tags.group_by("tag")
    .agg(pl.len().alias("istifade_sayi"))
    .sort("istifade_sayi", descending=True)
    .head(20)
)

In [98]:
print(top_20_teq)

shape: (20, 2)
┌───────────────────┬───────────────┐
│ tag               ┆ istifade_sayi │
│ ---               ┆ ---           │
│ str               ┆ u32           │
╞═══════════════════╪═══════════════╡
│ In Netflix queue  ┆ 131           │
│ atmospheric       ┆ 36            │
│ thought-provoking ┆ 24            │
│ superhero         ┆ 24            │
│ surreal           ┆ 23            │
│ …                 ┆ …             │
│ crime             ┆ 19            │
│ politics          ┆ 18            │
│ time travel       ┆ 16            │
│ music             ┆ 16            │
│ mental illness    ┆ 16            │
└───────────────────┴───────────────┘


### C3: For each movie, compute the proportion of its ratings that are 4 or higher, and print a table with movie ID, title, and high_rating_proportion (for movies with at least 5 ratings).

In [99]:
movie_high_ratings = ratings.group_by("movieId").agg(
    pl.len().alias("rating_count"),(pl.col("rating") >= 4).mean().alias("high_rating_proportion")
)

In [100]:
c3_result = (
    movie_high_ratings.filter(pl.col("rating_count") >= 5)
    .join(movies, on="movieId", how="inner")
    .select("movieId", "title", "high_rating_proportion")
    .sort("high_rating_proportion", descending=True) # Maraqlı olsun deyə yüksəkdən aşağı sıralayaq
)

In [101]:
print(c3_result)

shape: (3_650, 3)
┌─────────┬─────────────────────────────────┬────────────────────────┐
│ movieId ┆ title                           ┆ high_rating_proportion │
│ ---     ┆ ---                             ┆ ---                    │
│ i64     ┆ str                             ┆ f64                    │
╞═════════╪═════════════════════════════════╪════════════════════════╡
│ 177593  ┆ Three Billboards Outside Ebbin… ┆ 1.0                    │
│ 27156   ┆ Neon Genesis Evangelion: The E… ┆ 1.0                    │
│ 2940    ┆ Gilda (1946)                    ┆ 1.0                    │
│ 6460    ┆ Trial, The (Procès, Le) (1962)  ┆ 1.0                    │
│ 5008    ┆ Witness for the Prosecution (1… ┆ 1.0                    │
│ …       ┆ …                               ┆ …                      │
│ 2815    ┆ Iron Eagle (1986)               ┆ 0.0                    │
│ 51575   ┆ Wild Hogs (2007)                ┆ 0.0                    │
│ 3889    ┆ Highlander: Endgame (Highlande… ┆ 0.0          

### C4: For each user, compute the proportion of their ratings that are re-ratings (multiple ratings to the same movie), and print the top 10 users with the highest re-rating proportion.

In [102]:
user_re_ratings = ratings.group_by("userId").agg(
    pl.len().alias("total_ratings"),
    pl.col("movieId").n_unique().alias("unique_movies")
)

In [103]:
top_10_re_raters = (
    user_re_ratings.with_columns(
        ((pl.col("total_ratings") - pl.col("unique_movies")) / pl.col("total_ratings"))
        .alias("re_rating_proportion")
    )
    .sort("re_rating_proportion", descending=True)
    .head(10)
    .select("userId", "re_rating_proportion") # Yalnız tələb olunan sütunları göstəririk
)

In [104]:
print(top_10_re_raters)

shape: (10, 2)
┌────────┬──────────────────────┐
│ userId ┆ re_rating_proportion │
│ ---    ┆ ---                  │
│ i64    ┆ f64                  │
╞════════╪══════════════════════╡
│ 563    ┆ 0.0                  │
│ 554    ┆ 0.0                  │
│ 167    ┆ 0.0                  │
│ 218    ┆ 0.0                  │
│ 152    ┆ 0.0                  │
│ 16     ┆ 0.0                  │
│ 176    ┆ 0.0                  │
│ 183    ┆ 0.0                  │
│ 471    ┆ 0.0                  │
│ 451    ┆ 0.0                  │
└────────┴──────────────────────┘


### C5: Print the top 20 movies with the highest average ratings (display movie IDs, titles, and average ratings, with at least 10 ratings per movie).

In [105]:
movie_quality_stats = ratings.group_by("movieId").agg(
    pl.len().alias("rating_count"),
    pl.col("rating").mean().alias("avg_rating")
)

In [106]:
top_20_highest_rated = (
    movie_quality_stats.filter(pl.col("rating_count") >= 10)
    .join(movies, on="movieId", how="inner")
    .select("movieId", "title", "avg_rating")
    .sort("avg_rating", descending=True)
    .head(20)
)

In [107]:
print(top_20_highest_rated)


shape: (20, 3)
┌─────────┬─────────────────────────────────┬────────────┐
│ movieId ┆ title                           ┆ avg_rating │
│ ---     ┆ ---                             ┆ ---        │
│ i64     ┆ str                             ┆ f64        │
╞═════════╪═════════════════════════════════╪════════════╡
│ 1041    ┆ Secrets & Lies (1996)           ┆ 4.590909   │
│ 3451    ┆ Guess Who's Coming to Dinner (… ┆ 4.545455   │
│ 1178    ┆ Paths of Glory (1957)           ┆ 4.541667   │
│ 1104    ┆ Streetcar Named Desire, A (195… ┆ 4.475      │
│ 2360    ┆ Celebration, The (Festen) (199… ┆ 4.458333   │
│ …       ┆ …                               ┆ …          │
│ 7156    ┆ Fog of War: Eleven Lessons fro… ┆ 4.307692   │
│ 1209    ┆ Once Upon a Time in the West (… ┆ 4.305556   │
│ 92535   ┆ Louis C.K.: Live at the Beacon… ┆ 4.3        │
│ 475     ┆ In the Name of the Father (199… ┆ 4.3        │
│ 1204    ┆ Lawrence of Arabia (1962)       ┆ 4.3        │
└─────────┴──────────────────────────────

### C6: Compute and print the correlation between the number of ratings a movie receives and its average rating.

In [108]:
movie_stats = ratings.group_by("movieId").agg(
    pl.len().alias("rating_count"),
    pl.col("rating").mean().alias("avg_rating")
)

In [109]:
correlation_value = movie_stats.select(
    pl.corr("rating_count", "avg_rating")
).item()

In [110]:
print(f"Reytinq sayı ilə orta reytinq arasındakı korrelyasiya əmsalı: {correlation_value:.4f}")

Reytinq sayı ilə orta reytinq arasındakı korrelyasiya əmsalı: 0.1273
